# **Reading and Writing Data; Working with Text**

**DS112 — Computer Programming**

**Week 13:** · **Suggested class time: 5 hours (2-hour lecture + 3-hour laboratory)**

**ILO :** Use pandas to read and write CSV and Excel files and to process text data.

Most data arrives in **files**. pandas can read dozens of formats with one function call and turns them into DataFrames ready for analysis. This lesson also introduces pandas' tools for **text data**, using the course's collection of hotel reviews.

| Format | Read | Write |
| ------ | ---- | ----- |
| CSV | `pd.read_csv()` | `df.to_csv()` |
| Excel | `pd.read_excel()` | `df.to_excel()` |
| JSON | `pd.read_json()` | `df.to_json()` |
| HTML tables, SQL, Parquet, … | `pd.read_html()`, `pd.read_sql()`, `pd.read_parquet()` | … |

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

## **Reading CSV Files**

In [ ]:
grades = pd.read_csv("data/grades.csv")
grades.head()

> The `grades.csv`, `menu.csv`, `canteen_sales.csv`, and `weather.csv` files used in these lessons are **synthetic** datasets generated for this book.

### **Useful `read_csv` options**

| Option | Purpose | Example |
| ------ | ------- | ------- |
| `usecols` | read only some columns | `usecols=["name", "final"]` |
| `index_col` | use a column as the row labels | `index_col="student_id"` |
| `dtype` | force column types | `dtype={"section": "category"}` |
| `parse_dates` | convert columns to dates | `parse_dates=["date"]` |
| `na_values` | extra markers for missing data | `na_values=["n/a", "-"]` |
| `nrows` | read only the first rows | `nrows=100` |
| `sep` | a different separator | `sep=";"` |

In [ ]:
grades = pd.read_csv("data/grades.csv", index_col="student_id", dtype={"section": "category"})
print(grades.shape)
print(grades.isna().sum())          # missing values per column

In [ ]:
weather = pd.read_csv("data/weather.csv", parse_dates=["date"])
print(weather.dtypes)
weather.head(3)

## **Reading Excel Files**

`read_excel()` reads `.xlsx` files (it needs the `openpyxl` package). Use `sheet_name` to choose a sheet.

In [ ]:
loans = pd.read_excel("data/sample_data.xlsx")
print(loans.shape)
loans.head()

## **Writing Files**

Use `to_csv()` and `to_excel()`. Pass `index=False` unless the index carries information. The cell below writes to a **temporary** folder.

In [ ]:
import tempfile
from pathlib import Path

out = Path(tempfile.mkdtemp())
summary = grades.groupby("section", observed=True)[["quiz", "final"]].mean().round(2)

summary.to_csv(out / "section_summary.csv")
summary.to_excel(out / "section_summary.xlsx")
print(sorted(p.name for p in out.iterdir()))
print((out / "section_summary.csv").read_text())

## **Working with Text Data**

Text columns have a special **`.str` accessor** that applies string methods to **every** value at once — the vectorized version of the string methods from Week 7.

In [ ]:
reviews = pd.read_csv("data/reviews.csv", usecols=["id", "reviews.text"])
reviews = reviews.rename(columns={"reviews.text": "text"})
print(reviews.shape)
reviews.head(3)

In [ ]:
text = reviews["text"]

print(text.str.len().describe().round(1))          # review length in characters
print(text.str.lower().str.contains("breakfast").mean().round(3), "of reviews mention breakfast")
print(text.str.split().str.len().head())           # number of words

| Method | Purpose |
| ------ | ------- |
| `.str.lower()`, `.str.upper()`, `.str.strip()` | clean case and spaces |
| `.str.len()` | length of each string |
| `.str.contains(pattern)` | `True` if the text contains a word or pattern |
| `.str.replace(pattern, new, regex=True)` | substitute using a **regular expression** |
| `.str.split()`, `.str.get(i)` | split into lists / take an item |
| `.str.count(pattern)` | count matches |
| `.str.extract(pattern)` | pull out matching parts into a column |

### **A text-cleaning pipeline**

Before analysis, text is usually **normalized**: lowercase, remove punctuation and digits, and collapse extra spaces.

In [ ]:
clean = (
    reviews["text"]
    .str.lower()
    .str.replace(r"[^a-z\s]", " ", regex=True)      # keep letters and spaces only
    .str.replace(r"\s+", " ", regex=True)           # collapse repeated spaces
    .str.strip()
)
reviews = reviews.assign(clean=clean, words=clean.str.split().str.len())
reviews[["text", "clean", "words"]].head(3)

In [ ]:
words = reviews["clean"].str.split().explode()                # one row per word
stop = {"the", "and", "a", "to", "was", "i", "we", "is", "of", "in", "it", "for", "our", "my",
        "with", "were", "very", "but", "at", "this", "had", "not", "you", "on", "they", "that",
        "are", "there", "room", "hotel", "stay", "be", "as", "so", "would", "s", "t", "more"}
words[~words.isin(stop)].value_counts().head(10)

Positive and negative words can be counted with `.str.count()`:

In [ ]:
reviews["positive"] = reviews["clean"].str.count(r"\b(?:great|clean|friendly|excellent|comfortable)\b")
reviews["negative"] = reviews["clean"].str.count(r"\b(?:dirty|rude|noisy|broken|smell)\b")
print(reviews[["positive", "negative"]].sum())
reviews.sort_values("negative", ascending=False)[["text", "negative"]].head(2)

:::{admonition} Paradigm lens · functional
:class: tip

The cleaning code is a **chain of transformations** — `lower → replace → replace → strip` — each producing a new Series from the previous one, with no loop and no changing variables. This **method chaining** style is the functional pipeline from Week 3, applied to 10,000 reviews at once.
:::

## **Summary**

| Task | Code |
| ---- | ---- |
| read CSV / Excel | `pd.read_csv(path, …)`, `pd.read_excel(path, sheet_name=…)` |
| write CSV / Excel | `df.to_csv(path, index=False)`, `df.to_excel(path)` |
| count missing values | `df.isna().sum()` |
| vectorized text methods | `s.str.lower()`, `.str.contains()`, `.str.replace(regex=True)` |
| one row per word | `s.str.split().explode()` |
| most common values | `value_counts()` |

## **References**

Maulion, M. R. (2022). *DS 112 computer programming* [Instructional material]. University of Science and Technology of Southern Philippines.

McKinney, W. (2022). *Python for data analysis: Data wrangling with pandas, NumPy, and Jupyter* (3rd ed.). O'Reilly Media. https://wesmckinney.com/book/

The pandas development team. (2026). *IO tools (text, CSV, HDF5, …)*. https://pandas.pydata.org/docs/user_guide/io.html

The pandas development team. (2026). *Working with text data*. https://pandas.pydata.org/docs/user_guide/text.html

VanderPlas, J. (2023). *Python data science handbook: Essential tools for working with data* (2nd ed.). O'Reilly Media. https://jakevdp.github.io/PythonDataScienceHandbook/